In [2]:
!pip install langchain langchain-text-splitters --break-system-packages -q

# Stage 5 — company financial metrics -> business.company_metrics

Downloads each insurer's annual report, then combines a **deterministic
pass** (regex against a "Summary of Financial Statements" / "Financial
Highlights" block — reliable for GWP, GDPI, net worth, solvency ratio,
ROE, ROA) with a **retrieval + LLM pass** (FAISS + qwen2.5:7b via Ollama,
used only for the handful of metrics — like AUM — that aren't in a single
tidy table and need to be found by meaning rather than by a fixed label).

The regex pass is trusted first because it either finds an exact labelled
number or finds nothing; the LLM pass is used sparingly, and only for
metrics explicitly named in `tier2_metrics`, since a model asked to find
a number it can't find will sometimes produce a plausible-looking wrong
one instead of admitting it isn't there.

Two cells, run separately:

1. **All insurers except ABHI** — each company's annual report is its own
   PDF, so the whole document is a fair candidate for the summary block.
2. **ABHI only** — Aditya Birla's health arm doesn't publish a standalone
   annual report; its numbers are one section inside its parent's combined
   subsidiaries report (1,000+ pages), so the extraction is narrowed to a
   known page range and gated behind a data-quality check before it writes
   anything, so a wrong page range fails loudly instead of inserting
   garbage.

Cell outputs below are kept from the last real run, as a record of what
each company's extracted figures looked like.


## Cell 1 — all insurers except ABHI

In [ ]:
import json
import os
import re
import time
import requests
import pdfplumber
import ollama
import pyodbc
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# ==========================================================
# CONFIG
# ==========================================================

COMPANIES_ROOT = r"C:\Users\aksha\OneDrive\Desktop\insurance\Companies"
LATEST_FY_YEAR = 2025
SKIP_COMPANIES = ["ABHI"]  # different document type — not a standalone insurer annexure

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=AKSHAT\\SQLEXPRESS;"
    "DATABASE=INSURANCEDB;"
    "Trusted_Connection=yes;"
)

embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# ==========================================================
# DOWNLOAD HELPERS
# ==========================================================

def is_valid_pdf(file_path):
    try:
        with open(file_path, "rb") as f:
            header = f.read(5)
        return header == b"%PDF-"
    except Exception:
        return False

def download_via_requests(url, save_path):
    resp = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    with open(save_path, "wb") as f:
        f.write(resp.content)
    return is_valid_pdf(save_path)

def download_via_selenium(url, save_path, wait_seconds=5):
    # falls back here when a plain request gets blocked — a real browser
    # session follows the site's redirect/JS chain to the actual PDF URL
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    driver = webdriver.Chrome(options=chrome_options)
    try:
        driver.get(url)
        time.sleep(wait_seconds)
        final_url = driver.current_url
    finally:
        driver.quit()
    resp = requests.get(final_url, timeout=60, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })
    resp.raise_for_status()
    with open(save_path, "wb") as f:
        f.write(resp.content)
    return is_valid_pdf(save_path)

def download_pdf_smart(url, save_path):
    """Plain requests first (fast); Selenium only if that fails."""
    print("  Attempting standard download (requests)...")
    try:
        if download_via_requests(url, save_path):
            print("  ✅ Valid PDF via requests")
            return True
        print("  ⚠️  Not a valid PDF — retrying with Selenium")
    except Exception as e:
        print(f"  ⚠️  requests failed ({e}) — retrying with Selenium")
    try:
        if download_via_selenium(url, save_path):
            print("  ✅ Valid PDF via Selenium")
            return True
        print("  ❌ Selenium also produced a non-PDF file")
        return False
    except Exception as e:
        print(f"  ❌ Selenium failed: {e}")
        return False

# ==========================================================
# PIPELINE HELPERS
# ==========================================================

def get_company_id(conn, company_name):
    cursor = conn.cursor()
    cursor.execute("SELECT company_id FROM business.company_master WHERE company_name = ?", company_name)
    row = cursor.fetchone()
    cursor.close()
    return row[0] if row else None

def build_pipeline_for_pdf(pdf_path):
    """Extracts full text, then builds a small FAISS index over it for the
    handful of metrics (e.g. AUM) the regex pass below can't reliably find."""
    with pdfplumber.open(pdf_path) as pdf:
        all_text = ""
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                all_text += t + "\n"
    if len(all_text) < 500:
        raise ValueError("Extracted text too short — file may not be a valid text PDF")
    splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=400)
    chunks = splitter.split_text(all_text)
    vector_db = FAISS.from_texts(chunks, embedding_model)
    retriever = vector_db.as_retriever(search_kwargs={"k": 15})
    return all_text, retriever

def find_real_summary_block(full_text, size=8000):
    """
    Annual reports usually repeat the phrase "Financial Highlights" in a
    table of contents and section headers before the actual numbers table —
    so the first match isn't trustworthy. This walks every occurrence of
    each marker and keeps the first one that's actually followed by
    premium-shaped numbers, which is the real data block.
    """
    markers = [
        "SUMMARY OF FINANCIAL STATEMENTS",
        "Summary of Financial Statements",
        "FINANCIAL HIGHLIGHTS",
        "Financial Highlights",
    ]
    for marker in markers:
        start = 0
        while True:
            idx = full_text.lower().find(marker.lower(), start)
            if idx == -1:
                break
            chunk = full_text[idx: idx + size]
            if re.search(r'Gross\s*(Direct|Written)', chunk, re.IGNORECASE) and re.search(r'\d{3,}', chunk):
                return chunk, marker
            start = idx + 1
    return None, None

def detect_unit_divisor(block):
    """Figures are reported in lakhs or crores depending on the insurer;
    everything downstream is normalised to crores."""
    header = block[:400].lower()
    if "in lakh" in header or "₹ in lakhs" in header or "` in lakhs" in header:
        return 100.0
    if "in crore" in header or "₹ in crore" in header or "` in crores" in header:
        return 1.0
    return 100.0

def parse_row_multi(labels, text):
    """Finds a metric's row by label, then reads the 2–5 numbers that
    follow it as one financial-year series (oldest labels are tried in
    order so a synonym still matches if the primary label doesn't)."""
    NUM = r'\(?-?[\d,]+\.?\d*\)?'
    for label in labels:
        pattern = re.escape(label) + r'\s*((?:' + NUM + r'\s*){2,5})'
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            raw_nums = re.findall(NUM, m.group(1))
            nums = []
            for n in raw_nums:
                n = n.replace(',', '').replace('(', '-').replace(')', '')
                try:
                    nums.append(float(n))
                except ValueError:
                    pass
            if nums:
                return nums
    return None

def extract_market_share(all_text):
    """Excludes "market share" mentions that describe the industry as a
    whole (e.g. "private sector market share") rather than this insurer's
    own figure."""
    candidates = []
    for m in re.finditer(r'(\d+\.?\d*)%\s*market\s*share', all_text, re.IGNORECASE):
        window = all_text[max(0, m.start()-150): m.start()]
        if re.search(r'private sector|industry|insurers retained|overall', window, re.IGNORECASE):
            continue
        candidates.append(float(m.group(1)))
    m2 = re.search(r'(\d+\.?\d*)%\s*market share considering premiums', all_text, re.IGNORECASE)
    if m2:
        return float(m2.group(1))
    return candidates[0] if candidates else None

def extract_single_metric(retriever, search_queries, description):
    """
    Retrieval + LLM extraction for metrics that don't sit in one tidy
    table row. The prompt tells the model to omit anything it isn't
    confident about — a wrong guess is worse than a missing value here,
    since a missing value is visibly null while a wrong one looks real.
    """
    seen = set()
    chunks = []
    for q in search_queries:
        for doc in retriever.invoke(q):
            key = doc.page_content[:80]
            if key not in seen:
                seen.add(key)
                chunks.append(doc.page_content)
    ctx = "\n\n---\n\n".join(chunks)[:8000]
    schema = {
        "type": "object",
        "properties": {
            "years": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "financial_year": {"type": "string"},
                        "value": {"type": "number"}
                    },
                    "required": ["financial_year"]
                }
            }
        },
        "required": ["years"]
    }
    prompt = f"""
Extract ONLY the metric "{description}" from the context below, for every financial year mentioned.
Only include a value if it is EXPLICITLY stated as a number next to this exact metric name or a clear synonym.
If not confident it belongs to "{description}", omit it — do not guess.

Context:
{ctx}
"""
    response = ollama.chat(
        model="qwen2.5:7b",
        messages=[{"role": "user", "content": prompt}],
        format=schema,
        options={"temperature": 0}
    )
    try:
        data = json.loads(response["message"]["content"])
        return data.get("years", [])
    except Exception:
        return []

def extract_company_metrics(all_text, retriever, latest_fy_year):
    """Combines the deterministic regex pass (Tier 1: gwp, gdpi, net worth,
    solvency, roe, roa, market share) with the LLM pass (Tier 2: metrics
    like AUM that need semantic search) into one record per financial year."""
    years = [f"FY{str(latest_fy_year - i)[-2:]}" for i in range(5)]
    records = {y: {"financial_year": y} for y in years}

    block, matched_marker = find_real_summary_block(all_text)

    if block:
        divisor = detect_unit_divisor(block)
        print(f"  Matched marker: '{matched_marker}' | unit divisor: {divisor}")

        gwp = parse_row_multi(["Gross Written Premium"], block)
        gdpi = parse_row_multi(["Gross Direct Premium", "Gross Direct Written Premium", "Gross Direct Written"], block)
        net_worth = parse_row_multi(["Net worth", "Net Worth"], block)
        solvency = parse_row_multi(["Solvency Ratio"], block)
        pat = parse_row_multi(["Profit / (Loss) after tax", "Profit After Tax", "Profit/(Loss) after Tax"], block)
        assets = parse_row_multi(["Total Assets"], block)

        for i, y in enumerate(years):
            if gwp and i < len(gwp): records[y]["gwp"] = round(gwp[i]/divisor, 2)
            if gdpi and i < len(gdpi): records[y]["gdpi"] = round(gdpi[i]/divisor, 2)
            if net_worth and i < len(net_worth): records[y]["net_worth"] = round(net_worth[i]/divisor, 2)
            if solvency and i < len(solvency): records[y]["solvency_ratio"] = solvency[i]
            # ROE/ROA are derived, not read directly — the report states
            # profit-after-tax but not the ratio itself
            if pat and net_worth and i < len(pat) and records[y].get("net_worth"):
                records[y]["roe"] = round((pat[i]/divisor) / records[y]["net_worth"] * 100, 2)
            if pat and assets and i < len(pat) and i < len(assets) and assets[i]:
                records[y]["roa"] = round((pat[i]/divisor) / (assets[i]/divisor) * 100, 2)
    else:
        print("  WARNING: No summary/highlights block found — Tier 1 metrics empty")

    ms = extract_market_share(all_text)
    records[years[0]]["market_share"] = ms

    # Tier 2 — only metrics named here go through the slower LLM path
    tier2_metrics = {
        "aum": (["Assets Under Management investment portfolio"], "Assets Under Management"),
    }
    for field, (queries, desc) in tier2_metrics.items():
        results = extract_single_metric(retriever, queries, desc)
        for item in results:
            fy_raw = item.get("financial_year", "").upper().replace(" ", "")
            matched = next((y for y in years if y in fy_raw or y.replace("FY","") in fy_raw), None)
            if matched and item.get("value") is not None:
                records[matched][field] = item["value"]

    return records

def insert_metrics(conn, company_id, records):
    """MERGE rather than INSERT so re-running after a better extraction
    overwrites the same (company, financial_year) row instead of duplicating it."""
    cursor = conn.cursor()
    fields = ["gwp","gdpi","csr","icr","roe","roa","solvency_ratio",
              "market_share","net_worth","aum","combined_ratio","claim_turnaround_time"]
    for fy, row in records.items():
        values = [row.get(f) for f in fields]
        cursor.execute(f"""
            MERGE INTO business.company_metrics AS target
            USING (SELECT ? AS company_id, ? AS financial_year) AS src
            ON target.company_id = src.company_id AND target.financial_year = src.financial_year
            WHEN MATCHED THEN UPDATE SET
                gwp=?, gdpi=?, csr=?, icr=?, roe=?, roa=?, solvency_ratio=?,
                market_share=?, net_worth=?, aum=?, combined_ratio=?, claim_turnaround_time=?
            WHEN NOT MATCHED THEN INSERT
                (company_id, financial_year, gwp, gdpi, csr, icr, roe, roa,
                 solvency_ratio, market_share, net_worth, aum, combined_ratio, claim_turnaround_time)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
        """, (company_id, fy, *values, company_id, fy, *values))
    conn.commit()
    cursor.close()

# ==========================================================
# MAIN LOOP
# ==========================================================

company_folders = [f for f in os.listdir(COMPANIES_ROOT)
                    if os.path.isdir(os.path.join(COMPANIES_ROOT, f))
                    and f not in SKIP_COMPANIES]
print(f"Processing {len(company_folders)} companies: {company_folders}")

for folder in company_folders:
    folder_path = os.path.join(COMPANIES_ROOT, folder)
    links_file = os.path.join(folder_path, "pdf_links.json")

    if not os.path.exists(links_file):
        print(f"⚠️  Skipping {folder} — no pdf_links.json found")
        continue

    print(f"\n{'='*80}\nProcessing: {folder}\n{'='*80}")

    try:
        with open(links_file, "r") as f:
            company_data = json.load(f)

        company_name = company_data["company"]
        annual_report = next((p for p in company_data["pdfs"] if p["type"] == "AnnualReport"), None)
        if not annual_report or not annual_report.get("url"):
            print(f"⚠️  No valid AnnualReport URL for {company_name} — skipping")
            continue

        company_id = get_company_id(conn, company_name)
        if company_id is None:
            print(f"⚠️  '{company_name}' not found in business.company_master — skipping insert")
            continue

        download_dir = os.path.join(folder_path, "downloaded")
        os.makedirs(download_dir, exist_ok=True)
        local_pdf_path = os.path.join(download_dir, "annual_report.pdf")

        if os.path.exists(local_pdf_path) and is_valid_pdf(local_pdf_path):
            print("  Using previously downloaded PDF")
        else:
            success = download_pdf_smart(annual_report["url"], local_pdf_path)
            if not success:
                print(f"❌ Could not download a valid PDF for {company_name} — skipping")
                continue

        all_text, retriever = build_pipeline_for_pdf(local_pdf_path)
        print(f"  Extracted text length: {len(all_text)}")

        records = extract_company_metrics(all_text, retriever, LATEST_FY_YEAR)
        print(json.dumps(records, indent=2))
        insert_metrics(conn, company_id, records)

        print(f"✅ {company_name} inserted (company_id={company_id})")

    except Exception as e:
        print(f"❌ FAILED for {folder}: {e}")

conn.close()
print("\nAll companies processed.")

## Cell 2 — ABHI only

ABHI's health insurance financials live inside pages ~400–500 of its
parent company's combined subsidiaries report, not in a standalone annual
report, so this repeats cell 1's extraction logic but narrowed to that
page range and gated behind a data-quality check (`is_satisfactory`)
before anything is written to SQL.

In [ ]:
import json
import os
import re
import time
import requests
import pdfplumber
import ollama
import pyodbc
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# ==========================================================
# CONFIG — ABHI ONLY
# ==========================================================

COMPANY_NAME = "ABHI"
COMPANY_DIR = r"C:\Users\aksha\OneDrive\Desktop\insurance\Companies\ABHI"
DOWNLOAD_DIR = os.path.join(COMPANY_DIR, "downloaded")
LATEST_FY_YEAR = 2025

PDF_URL = "https://www.adityabirlacapital.com/-/media/ABCL/pdf/Financial-reports-of-our-subsidiary-companies/Aditya-Birla-Subsidiaries-Financial-Report-24-25_-Web-High-Reso.webp?extension=webp"

# Page range where the financial data lives (0-indexed for pdfplumber).
# Found by manual inspection of the combined subsidiaries report — if
# Aditya Birla reshuffles the document in a future filing, re-check this.
PAGE_START = 395   # buffer before 400
PAGE_END = 505     # buffer after 500

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=AKSHAT\\SQLEXPRESS;"
    "DATABASE=INSURANCEDB;"
    "Trusted_Connection=yes;"
)

# ==========================================================
# STEP 1 — DOWNLOAD
# ==========================================================

def is_valid_pdf(file_path):
    try:
        with open(file_path, "rb") as f:
            header = f.read(5)
        return header == b"%PDF-"
    except Exception:
        return False

def download_via_requests(url, save_path):
    resp = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    with open(save_path, "wb") as f:
        f.write(resp.content)
    return is_valid_pdf(save_path)

def download_via_selenium(url, save_path, wait_seconds=6):
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    driver = webdriver.Chrome(options=chrome_options)
    try:
        driver.get(url)
        time.sleep(wait_seconds)
        final_url = driver.current_url
    finally:
        driver.quit()
    resp = requests.get(final_url, timeout=90, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })
    resp.raise_for_status()
    with open(save_path, "wb") as f:
        f.write(resp.content)
    return is_valid_pdf(save_path)

def download_pdf_smart(url, save_path):
    print("Attempting standard download (requests)...")
    try:
        if download_via_requests(url, save_path):
            print("✅ Valid PDF via requests")
            return True
        print("⚠️  Not a valid PDF — retrying with Selenium")
    except Exception as e:
        print(f"⚠️  requests failed ({e}) — retrying with Selenium")
    try:
        if download_via_selenium(url, save_path):
            print("✅ Valid PDF via Selenium")
            return True
        print("❌ Selenium also produced a non-PDF file")
        return False
    except Exception as e:
        print(f"❌ Selenium failed: {e}")
        return False

local_pdf_path = os.path.join(DOWNLOAD_DIR, "abhi_subsidiary_report.pdf")

if os.path.exists(local_pdf_path) and is_valid_pdf(local_pdf_path):
    print("Using previously downloaded PDF")
else:
    success = download_pdf_smart(PDF_URL, local_pdf_path)
    if not success:
        raise RuntimeError("Could not download a valid PDF for ABHI — check URL manually")

print("File size:", os.path.getsize(local_pdf_path), "bytes")

# ==========================================================
# STEP 2 — EXTRACT ONLY THE RELEVANT PAGE RANGE (400-500)
# ==========================================================
# the full report is 1000+ pages; extracting and embedding all of it would
# be slow and would dilute retrieval with unrelated subsidiaries' figures

def extract_page_range_text(pdf_path, start_page, end_page):
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"Total pages in PDF: {total_pages}")
        actual_start = max(0, start_page - 1)  # convert to 0-indexed
        actual_end = min(total_pages, end_page)
        print(f"Extracting pages {actual_start+1} to {actual_end} (0-indexed {actual_start}-{actual_end})")

        for i in range(actual_start, actual_end):
            t = pdf.pages[i].extract_text()
            if t:
                all_text += t + "\n"
    return all_text

all_text = extract_page_range_text(local_pdf_path, PAGE_START, PAGE_END)
print("Extracted text length:", len(all_text))

if len(all_text) < 500:
    raise ValueError("Extracted text too short — page range may be wrong, or these pages are scanned/image-based")

# ==========================================================
# STEP 3 — BUILD RAG PIPELINE (chunk_size=2500, overlap=400 as requested)
# ==========================================================

splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=400)
chunks = splitter.split_text(all_text)
print(f"Created {len(chunks)} chunks")

vector_db = FAISS.from_texts(chunks, embedding_model)
retriever = vector_db.as_retriever(search_kwargs={"k": 15})

# ==========================================================
# STEP 4 — DETERMINISTIC + LLM EXTRACTION (same logic as cell 1)
# ==========================================================

def find_real_summary_block(full_text, size=8000):
    markers = [
        "SUMMARY OF FINANCIAL STATEMENTS",
        "Summary of Financial Statements",
        "FINANCIAL HIGHLIGHTS",
        "Financial Highlights",
    ]
    for marker in markers:
        start = 0
        while True:
            idx = full_text.lower().find(marker.lower(), start)
            if idx == -1:
                break
            chunk = full_text[idx: idx + size]
            if re.search(r'Gross\s*(Direct|Written)', chunk, re.IGNORECASE) and re.search(r'\d{3,}', chunk):
                return chunk, marker
            start = idx + 1
    return None, None

def detect_unit_divisor(block):
    header = block[:400].lower()
    if "in lakh" in header or "₹ in lakhs" in header or "` in lakhs" in header:
        return 100.0
    if "in crore" in header or "₹ in crore" in header or "` in crores" in header:
        return 1.0
    return 100.0

def parse_row_multi(labels, text):
    NUM = r'\(?-?[\d,]+\.?\d*\)?'
    for label in labels:
        pattern = re.escape(label) + r'\s*((?:' + NUM + r'\s*){2,5})'
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            raw_nums = re.findall(NUM, m.group(1))
            nums = []
            for n in raw_nums:
                n = n.replace(',', '').replace('(', '-').replace(')', '')
                try:
                    nums.append(float(n))
                except ValueError:
                    pass
            if nums:
                return nums
    return None

def extract_market_share(all_text):
    candidates = []
    for m in re.finditer(r'(\d+\.?\d*)%\s*market\s*share', all_text, re.IGNORECASE):
        window = all_text[max(0, m.start()-150): m.start()]
        if re.search(r'private sector|industry|insurers retained|overall', window, re.IGNORECASE):
            continue
        candidates.append(float(m.group(1)))
    return candidates[0] if candidates else None

def extract_company_metrics(all_text, latest_fy_year):
    years = [f"FY{str(latest_fy_year - i)[-2:]}" for i in range(5)]
    records = {y: {"financial_year": y} for y in years}

    block, matched_marker = find_real_summary_block(all_text)

    if block:
        divisor = detect_unit_divisor(block)
        print(f"Matched marker: '{matched_marker}' | unit divisor: {divisor}")

        gwp = parse_row_multi(["Gross Written Premium"], block)
        gdpi = parse_row_multi(["Gross Direct Premium", "Gross Direct Written Premium", "Gross Direct Written"], block)
        net_worth = parse_row_multi(["Net worth", "Net Worth"], block)
        solvency = parse_row_multi(["Solvency Ratio"], block)
        pat = parse_row_multi(["Profit / (Loss) after tax", "Profit After Tax", "Profit/(Loss) after Tax"], block)
        assets = parse_row_multi(["Total Assets"], block)

        for i, y in enumerate(years):
            if gwp and i < len(gwp): records[y]["gwp"] = round(gwp[i]/divisor, 2)
            if gdpi and i < len(gdpi): records[y]["gdpi"] = round(gdpi[i]/divisor, 2)
            if net_worth and i < len(net_worth): records[y]["net_worth"] = round(net_worth[i]/divisor, 2)
            if solvency and i < len(solvency): records[y]["solvency_ratio"] = solvency[i]
            if pat and net_worth and i < len(pat) and records[y].get("net_worth"):
                records[y]["roe"] = round((pat[i]/divisor) / records[y]["net_worth"] * 100, 2)
            if pat and assets and i < len(pat) and i < len(assets) and assets[i]:
                records[y]["roa"] = round((pat[i]/divisor) / (assets[i]/divisor) * 100, 2)
    else:
        print("WARNING: No summary/highlights block found in this page range")

    ms = extract_market_share(all_text)
    records[years[0]]["market_share"] = ms

    return records

records = extract_company_metrics(all_text, LATEST_FY_YEAR)
print("\n" + "="*80)
print("EXTRACTED RECORDS")
print("="*80)
print(json.dumps(records, indent=2))

# ==========================================================
# STEP 5 — ONLY INSERT IF DATA QUALITY LOOKS ACCEPTABLE
# ==========================================================
# a wrong PAGE_START/PAGE_END would silently extract someone else's
# subsidiary numbers, so nothing is written unless at least the headline
# gwp figure was actually found

def is_satisfactory(records):
    """
    Require at least gwp present for the latest year to consider this a success.
    Adjust this bar if you want stricter/looser criteria.
    """
    latest = list(records.values())[0]
    return latest.get("gwp") is not None

if is_satisfactory(records):
    print("\n✅ Data quality check passed — inserting into SQL")

    cursor = conn.cursor()
    cursor.execute("SELECT company_id FROM business.company_master WHERE company_name = ?", COMPANY_NAME)
    row = cursor.fetchone()
    cursor.close()

    if row is None:
        print(f"⚠️  '{COMPANY_NAME}' not found in business.company_master — cannot insert")
    else:
        company_id = row[0]
        cursor = conn.cursor()
        fields = ["gwp","gdpi","csr","icr","roe","roa","solvency_ratio",
                  "market_share","net_worth","aum","combined_ratio","claim_turnaround_time"]
        for fy, rec in records.items():
            values = [rec.get(f) for f in fields]
            cursor.execute(f"""
                MERGE INTO business.company_metrics AS target
                USING (SELECT ? AS company_id, ? AS financial_year) AS src
                ON target.company_id = src.company_id AND target.financial_year = src.financial_year
                WHEN MATCHED THEN UPDATE SET
                    gwp=?, gdpi=?, csr=?, icr=?, roe=?, roa=?, solvency_ratio=?,
                    market_share=?, net_worth=?, aum=?, combined_ratio=?, claim_turnaround_time=?
                WHEN NOT MATCHED THEN INSERT
                    (company_id, financial_year, gwp, gdpi, csr, icr, roe, roa,
                     solvency_ratio, market_share, net_worth, aum, combined_ratio, claim_turnaround_time)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
            """, (company_id, fy, *values, company_id, fy, *values))
        conn.commit()
        cursor.close()
        print(f"✅ ABHI inserted (company_id={company_id})")
else:
    print("\n❌ Data quality check FAILED — gwp not found for latest year")
    print("Not inserting into SQL. Review the printed context/records above, or adjust PAGE_START/PAGE_END.")

conn.close()